# Detectores de layout no dataset do Label Studio (2 × T4)

Treina vários modelos nas **mesmas páginas e no mesmo split por fonte** do treino local, uma fila por GPU.

**Antes de rodar:** Settings → Accelerator **GPU T4 x2**, **Internet ON**, e o dataset `ls_deteccao` (PRIVADO) adicionado em *Add input*. Depois: **Save Version → Save & Run All (Commit)** e pode fechar a aba.

No fim, baixe `saida_ls.zip` na aba *Output*.

In [ ]:
!pip -q install -U ultralytics huggingface_hub 2>&1 | tail -1
!pip -q install doclayout-yolo 2>&1 | tail -1   # se falhar, as filas pulam esse modelo
import torch, glob, json, os
print('torch', torch.__version__, '| GPUs:', torch.cuda.device_count())
assert torch.cuda.device_count() >= 1, 'Ligue a GPU em Settings -> Accelerator'

In [ ]:
# --- monta os dois datasets (ls16 e ls3) em /kaggle/working/ds, com links para as imagens ---
achados = glob.glob('/kaggle/input/**/split.json', recursive=True)
assert achados, 'Dataset ls_deteccao nao encontrado em /kaggle/input (Add input)'
ENT = os.path.dirname(achados[0])
split = json.load(open(f'{ENT}/split.json'))
classes = json.load(open(f'{ENT}/classes.json'))
import shutil
for ds in ('ls16', 'ls3'):
    raiz = f'/kaggle/working/ds/{ds}'
    for img, sp in split.items():
        os.makedirs(f'{raiz}/images/{sp}', exist_ok=True); os.makedirs(f'{raiz}/labels/{sp}', exist_ok=True)
        dst = f'{raiz}/images/{sp}/{img}'
        if not os.path.lexists(dst): os.symlink(f'{ENT}/images/{img}', dst)
        stem = os.path.splitext(img)[0]
        shutil.copy(f'{ENT}/labels_{ds}/{stem}.txt', f'{raiz}/labels/{sp}/{stem}.txt')
    nomes = '\n'.join(f'  {i}: {c}' for i, c in enumerate(classes[ds]))
    open(f'{raiz}/data.yaml', 'w').write(f'path: {raiz}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n{nomes}\n')
from collections import Counter
print('paginas por split:', dict(Counter(split.values())))

In [ ]:
%%writefile /kaggle/working/treina.py
import sys, json, os, glob, shutil, traceback
ds, arq, imgsz, batch, horas, nome, dev = sys.argv[1:8]
os.environ["CUDA_VISIBLE_DEVICES"] = dev            # cada fila enxerga so a sua GPU
# o fork doclayout_yolo liga o wandb sozinho e usa o caminho de "project" como nome do projeto: no Kaggle
# de 03/10 os dois treinos do DocLayout cairam com "Invalid project name '/kaggle/working/runs'"
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"
# 08/10: so as variaveis nao bastam (a validacao do nome do projeto roda antes do "disabled"): desliga o callback
# do wandb pelas SETTINGS do fork e usa um "project" relativo, sem barra
try:
    from doclayout_yolo.utils import SETTINGS
    SETTINGS.update({"wandb": False})
except Exception as e:
    print("aviso: nao consegui desligar o wandb pelas SETTINGS:", e)
os.chdir("/kaggle/working")
imgsz, batch, horas = int(imgsz), int(batch), float(horas)
RAIZ = "/kaggle/working"
data = f"{RAIZ}/ds/{ds}/data.yaml"
saida = f"{RAIZ}/saida/{nome}"
os.makedirs(saida, exist_ok=True)
try:
    kw = dict(data=data, epochs=100, patience=20, imgsz=imgsz, batch=batch, device=0, seed=42,
              project="runs", name=nome, exist_ok=True, fliplr=0.0, workers=2)
    if arq == "doclayout":                           # pesos pre-treinados em documentos (DocStructBench)
        from doclayout_yolo import YOLOv10 as Modelo
        from huggingface_hub import hf_hub_download
        pesos = hf_hub_download("juliozhao/DocLayout-YOLO-DocStructBench",
                                "doclayout_yolo_docstructbench_imgsz1024.pt")
        kw["epochs"] = int(horas)                    # o fork nao tem limite por tempo: aqui "horas" = epocas
    elif arq.startswith("rtdetr"):
        from ultralytics import RTDETR as Modelo
        pesos, kw["time"] = arq, horas
    else:
        from ultralytics import YOLO as Modelo
        pesos, kw["time"] = arq, horas
    Modelo(pesos).train(**kw)

    best = f"{RAIZ}/runs/{nome}/weights/best.pt"
    m = Modelo(best)
    r = m.val(data=data, split="test", imgsz=imgsz, batch=batch, device=0, verbose=False,
              project="runs", name=nome + "_teste", exist_ok=True)
    met = {"modelo": arq, "dataset": ds, "imgsz": imgsz, "mAP50": float(r.box.map50), "mAP50_95": float(r.box.map),
           "precisao": float(r.box.mp), "revocacao": float(r.box.mr),
           "por_classe": {m.names[int(c)]: {"AP50": float(r.box.ap50[i]), "AP50_95": float(r.box.ap[i])}
                          for i, c in enumerate(r.box.ap_class_index)}}
    json.dump(met, open(f"{saida}/metricas_teste.json", "w"), ensure_ascii=False, indent=1)

    entrada = os.path.dirname(glob.glob("/kaggle/input/**/split.json", recursive=True)[0])
    with open(f"{saida}/pred_diamond_teste.jsonl", "w") as out:
        for img in sorted(glob.glob(f"{entrada}/diamond_teste/img/*.jpg")):
            res = m.predict(img, imgsz=imgsz, conf=0.25, device=0, verbose=False)[0]
            h, w = res.orig_shape
            regs = [{"classe": res.names[int(b.cls[0])], "bbox_px": [round(float(v), 1) for v in b.xyxy[0].tolist()],
                     "score": round(float(b.conf[0]), 4)} for b in res.boxes]
            out.write(json.dumps({"imagem": os.path.basename(img), "w": w, "h": h, "regioes": regs},
                                 ensure_ascii=False) + "\n")
    shutil.copy(best, f"{saida}/best.pt")
    shutil.copy(f"{RAIZ}/runs/{nome}/results.csv", f"{saida}/results.csv")
    print("FIM_OK", nome, met["mAP50"], met["mAP50_95"])
except Exception:
    open(f"{saida}/erro.txt", "w").write(traceback.format_exc())
    traceback.print_exc()
    print("FIM_ERRO", nome)


In [ ]:
# --- duas filas, uma por GPU. Colunas: dataset, modelo, imgsz, batch, horas (ou epocas p/ doclayout), nome ---
FILAS = {
  # 05/10: so os dois DocLayout-YOLO (os outros modelos ja rodaram em 03/10 e o fork nao liga mais o wandb)
  '0': [('ls3', 'doclayout', 1024, 8, 45, 'ls3_doclayout_1024')],
  '1': [('ls16', 'doclayout', 1024, 8, 45, 'ls16_doclayout_1024')],
}
import subprocess, time
if torch.cuda.device_count() < 2:      # so 1 GPU: tudo numa fila
    FILAS = {'0': FILAS['0'] + FILAS['1']}
os.makedirs('/kaggle/working/logs', exist_ok=True)
procs = []
for gpu, jobs in FILAS.items():
    cmd = ' ; '.join(f'python /kaggle/working/treina.py {d} {m} {i} {b} {h} {n} {gpu} > /kaggle/working/logs/{n}.log 2>&1'
                     for d, m, i, b, h, n in jobs)
    procs.append(subprocess.Popen(cmd, shell=True))
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(600)
    feitos = [os.path.basename(os.path.dirname(f)) for f in glob.glob('/kaggle/working/saida/*/metricas_teste.json')]
    print(f'{(time.time()-t0)/3600:.1f} h | concluidos: {feitos}', flush=True)
print('filas terminadas')

In [ ]:
# --- resumo e pacote de saida ---
import pandas as pd
linhas = []
for f in sorted(glob.glob('/kaggle/working/saida/*/metricas_teste.json')):
    m = json.load(open(f)); linhas.append({'nome': os.path.basename(os.path.dirname(f)), 'dataset': m['dataset'],
        'modelo': m['modelo'], 'imgsz': m['imgsz'], 'mAP50': round(m['mAP50'], 3), 'mAP50-95': round(m['mAP50_95'], 3)})
display(pd.DataFrame(linhas))
for f in glob.glob('/kaggle/working/saida/*/erro.txt'): print('ERRO em', f, '\n', open(f).read()[-600:])
shutil.copytree('/kaggle/working/logs', '/kaggle/working/saida/logs', dirs_exist_ok=True)
shutil.make_archive('/kaggle/working/saida_ls', 'zip', '/kaggle/working/saida')
print('Baixe /kaggle/working/saida_ls.zip na aba Output')